In [31]:
import polars as pl

# Define the path to the data file
file_path = "/media/m2_front/research/data/dexamine/uniswap_v3/usdc_weth_005/events_usdc_weth_005.parquet"

# Load data with Polars
try:
    data = pl.read_parquet("/media/m2_front/research/data/dexamine/uniswap_v3/usdc_weth_005/events_usdc_weth_005.parquet")
    print("Data loaded successfully. Here is an overview:")
    print(data)
    print("Here are the columns:")
    print(data.columns)
except FileNotFoundError:
    print(f"File not found: {file_path}")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

Data loaded successfully. Here is an overview:
shape: (7_704_825, 34)
┌────────────┬────────────┬───────┬────────────┬───┬────────────┬────────────┬───────────┬─────────┐
│ timestamp  ┆ block_numb ┆ index ┆ event_inde ┆ … ┆ virtual_re ┆ virtual_re ┆ to_type   ┆ tx_type │
│ ---        ┆ er         ┆ ---   ┆ x          ┆   ┆ serve_0    ┆ serve_1    ┆ ---       ┆ ---     │
│ i64        ┆ ---        ┆ i64   ┆ ---        ┆   ┆ ---        ┆ ---        ┆ str       ┆ i64     │
│            ┆ i64        ┆       ┆ i64        ┆   ┆ f64        ┆ f64        ┆           ┆         │
╞════════════╪════════════╪═══════╪════════════╪═══╪════════════╪════════════╪═══════════╪═════════╡
│ 1620250931 ┆ 12376729   ┆ 59    ┆ 5          ┆ … ┆ null       ┆ null       ┆ dex_route ┆ 0       │
│            ┆            ┆       ┆            ┆   ┆            ┆            ┆ r         ┆         │
│ 1620252901 ┆ 12376891   ┆ 72    ┆ 3          ┆ … ┆ 20129.1207 ┆ 5.915581   ┆ dex_route ┆ 0       │
│            ┆       

In [32]:
# Forward fill price column 
# "price" is the marginal price of the pool after the swap. "price" is not present in LP events,
# so we need to fill the missing values with the last observed price
print(data.select("marginal_price")[0:5])

data = data.with_columns(pl.col("marginal_price").fill_null(strategy="forward"))
print(data.select("marginal_price")[0:5])

print("Here are desciptive statistics:")
print(data.select("marginal_price").describe())

shape: (5, 1)
┌────────────────┐
│ marginal_price │
│ ---            │
│ f64            │
╞════════════════╡
│ null           │
│ 3402.729189    │
│ null           │
│ null           │
│ null           │
└────────────────┘
shape: (5, 1)
┌────────────────┐
│ marginal_price │
│ ---            │
│ f64            │
╞════════════════╡
│ null           │
│ 3402.729189    │
│ 3402.729189    │
│ 3402.729189    │
│ 3402.729189    │
└────────────────┘
Here are desciptive statistics:
shape: (9, 2)
┌────────────┬────────────────┐
│ statistic  ┆ marginal_price │
│ ---        ┆ ---            │
│ str        ┆ f64            │
╞════════════╪════════════════╡
│ count      ┆ 7.704824e6     │
│ null_count ┆ 1.0            │
│ mean       ┆ 2283.347482    │
│ std        ┆ 866.072886     │
│ min        ┆ 863.902755     │
│ 25%        ┆ 1599.24003     │
│ 50%        ┆ 1988.528016    │
│ 75%        ┆ 3002.538555    │
│ max        ┆ 4872.179753    │
└────────────┴────────────────┘


In [33]:
# Transform timestamp to date from unix time
data = data.with_columns(
    pl.from_epoch(pl.col("timestamp"), time_unit="s").alias("clocktime")
)

# Filter for dates after 2021-07-01
data = data.filter(pl.col("clocktime") >= pl.datetime(2021, 7, 1))

print(data.select("clocktime"))

shape: (7_514_865, 1)
┌─────────────────────┐
│ clocktime           │
│ ---                 │
│ datetime[μs]        │
╞═════════════════════╡
│ 2021-07-01 00:00:09 │
│ 2021-07-01 00:00:50 │
│ 2021-07-01 00:00:50 │
│ 2021-07-01 00:00:50 │
│ 2021-07-01 00:01:37 │
│ …                   │
│ 2024-09-19 15:25:11 │
│ 2024-09-19 15:25:35 │
│ 2024-09-19 15:25:47 │
│ 2024-09-19 15:26:59 │
│ 2024-09-19 15:27:11 │
└─────────────────────┘


In [34]:
# Add a new column 'return' that calculates the return based on 'price'
data = data.with_columns(
    ((pl.col("marginal_price") / pl.col("marginal_price").shift(1)) - 1).alias("return")
)

print(data.select("return").describe())


shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ return      │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 7.514864e6  │
│ null_count ┆ 1.0         │
│ mean       ┆ 3.8404e-7   │
│ std        ┆ 0.000865    │
│ min        ┆ -0.307077   │
│ 25%        ┆ -0.000015   │
│ 50%        ┆ -2.1729e-11 │
│ 75%        ┆ 0.000012    │
│ max        ┆ 0.211244    │
└────────────┴─────────────┘


In [35]:
# Filter large returns to examine them more closely
large_returns = data.filter((pl.col("return") > 0.15) | (pl.col("return") < -0.15))
print(large_returns)

# Convert to list and print all hashes
hashes = large_returns.select("tx_hash").to_series().to_list()
print(hashes)
usd_amounts = large_returns.select("amount_0").to_series().to_list()
print(usd_amounts)

# Filter for transactions with large returns around block 14953915
around_returns = data.filter((pl.col("block_number") < 14953918) & (pl.col("block_number") > 14953913))
print(around_returns)

shape: (2, 36)
┌────────────┬────────────┬───────┬────────────┬───┬────────────┬─────────┬────────────┬───────────┐
│ timestamp  ┆ block_numb ┆ index ┆ event_inde ┆ … ┆ to_type    ┆ tx_type ┆ clocktime  ┆ return    │
│ ---        ┆ er         ┆ ---   ┆ x          ┆   ┆ ---        ┆ ---     ┆ ---        ┆ ---       │
│ i64        ┆ ---        ┆ i64   ┆ ---        ┆   ┆ str        ┆ i64     ┆ datetime[μ ┆ f64       │
│            ┆ i64        ┆       ┆ i64        ┆   ┆            ┆         ┆ s]         ┆           │
╞════════════╪════════════╪═══════╪════════════╪═══╪════════════╪═════════╪════════════╪═══════════╡
│ 1655092586 ┆ 14953915   ┆ 0     ┆ 83         ┆ … ┆ smart_cont ┆ 2       ┆ 2022-06-13 ┆ -0.307077 │
│            ┆            ┆       ┆            ┆   ┆ ract       ┆         ┆ 03:56:26   ┆           │
│ 1655092628 ┆ 14953917   ┆ 0     ┆ 5          ┆ … ┆ mev        ┆ 2       ┆ 2022-06-13 ┆ 0.211244  │
│            ┆            ┆       ┆            ┆   ┆            ┆         ┆ 

In [36]:
# Filter for transactions around block 14953915
plot_data = data.filter((pl.col("block_number") < 14953930) & (pl.col("block_number") > 14953900))

# Plot the returns
import plotly.graph_objects as go
import polars as pl

# Assuming 'data' is your Polars DataFrame with 'timestamp' and 'return' columns
# Convert Polars columns to NumPy arrays
x_values = plot_data.select("clocktime").to_numpy().flatten()
y_values =plot_data.select("marginal_price").to_numpy().flatten()

# Create a figure and add Scattergl trace
fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=x_values,
        y=y_values,
        mode='markers',
        marker=dict(
            line=dict(
                width=1,
                color='DarkSlateGrey'
            )
        )
    )
)

# Display the figure
fig.show()

In [37]:
# Verify that LPs have zero impact on the marginal price (return)
lp_events = data.filter((pl.col("event_type") == "mint") | (pl.col("event_type") == "burn"))
print(lp_events.select("return").describe())

shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ return   │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 259673.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 0.0      │
│ std        ┆ 0.0      │
│ min        ┆ 0.0      │
│ 25%        ┆ 0.0      │
│ 50%        ┆ 0.0      │
│ 75%        ┆ 0.0      │
│ max        ┆ 0.0      │
└────────────┴──────────┘
